# What is in your traces?

**Question this notebook answers:** Before building any graph, you need to know
what raw material you have. This notebook describes the content of your agent
traces: which tools were called, how large a share is shell execution, how many
commands are in the corpus, and what time period they span.

This is not analysis — it is a sanity check. A corpus that is too small, or
dominated by a single tool, will produce a degenerate graph regardless of how
well the graph code is written.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Set SESSIONS_GLOB to the glob pattern that matches YOUR session files.
# The default works for OpenClaw traces. Adapt it to your agent format.

SESSIONS_GLOB = "~/.openclaw/agents/*/sessions/*.jsonl"

# Optional: filter to a specific agent name (leave empty to include all).
AGENT_FILTER = ""
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import glob
import os
import collections

from agentgraph import iter_tool_calls

# Expand the glob and fail early with a clear message if no files are found.
matched = glob.glob(os.path.expanduser(SESSIONS_GLOB))
if not matched:
    raise FileNotFoundError(
        f"No session files found for pattern: {SESSIONS_GLOB!r}\n"
        "Update SESSIONS_GLOB in the configuration cell above to point "
        "to your trace files."
    )
print(f"{len(matched)} session file(s) found.")

In [ ]:
# Load all tool calls.
calls = list(iter_tool_calls(SESSIONS_GLOB))

if AGENT_FILTER:
    calls = [c for c in calls if c.agent == AGENT_FILTER]
    print(f"Filtered to agent '{AGENT_FILTER}': {len(calls)} tool calls.")
else:
    print(f"Total tool calls: {len(calls)}")

if not calls:
    raise ValueError(
        "No tool calls found. Check that your session files contain "
        "events with a 'toolCall' type in their content blocks."
    )

In [ ]:
# Tool distribution — how often each tool was called.
tool_counts = collections.Counter(c.tool for c in calls)
total = sum(tool_counts.values())

print(f"{'Tool':<30} {'Count':>8} {'Share':>8}")
print("-" * 50)
for tool, count in tool_counts.most_common(20):
    print(f"{tool:<30} {count:>8}  {100 * count / total:>6.1f}%")
if len(tool_counts) > 20:
    print(f"... and {len(tool_counts) - 20} more tools.")

In [ ]:
# Shell execution share.
# The graph is built from shell commands, so if only a small fraction of calls
# are shell executions, the graph will cover a limited slice of agent activity.

SHELL_TOOLS = {"exec", "bash", "shell"}
shell_calls = [c for c in calls if c.tool in SHELL_TOOLS]
shell_pct = 100 * len(shell_calls) / total if total else 0

print(f"Shell tool calls : {len(shell_calls):>6}  ({shell_pct:.1f}% of all calls)")
print(f"Non-shell calls  : {total - len(shell_calls):>6}  ({100 - shell_pct:.1f}%)")

if shell_pct < 10:
    print()
    print("WARNING: fewer than 10% of calls are shell executions.")
    print("The graph will be sparse. Consider including other tool types.")

In [ ]:
# Time period and volume.
agents = collections.Counter(c.agent for c in calls)
sessions = collections.Counter(c.session for c in calls)

timestamps = sorted(c.timestamp for c in calls if c.timestamp)
first = timestamps[0][:10] if timestamps else "unknown"
last  = timestamps[-1][:10] if timestamps else "unknown"

print(f"Period        : {first} → {last}")
print(f"Agents        : {len(agents)}  → {dict(agents.most_common(5))}")
print(f"Sessions      : {len(sessions)}")
print(f"Tool calls    : {total}")
print(f"Shell calls   : {len(shell_calls)}")
print()
if not timestamps:
    print("WARNING: no timestamps found. Stability analysis (notebook 04) will not work.")

## How to read this

**Shell share below 10%** — the graph covers only a thin slice of agent
activity. The motifs and stability results will describe a minority of what
the agent actually does.

**A single agent dominates** — this is fine for graph analysis. But if
you plan to run stability analysis across agents, filter to one first
(set `AGENT_FILTER` above).

**Fewer than ~200 shell calls** — the graph will be too sparse to yield
meaningful motifs. Accumulate more traces before proceeding to notebooks
02–04.

**No timestamps** — update `iter_tool_calls()` in `agentgraph/extract.py`
to emit timestamps from your format. Without them, stability analysis is
impossible.

When you are satisfied with the corpus, continue to **notebook 02** to
find the right granularity for your graph.